In [2]:
import os
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from herbie import Herbie

# Desactivar advertencias de cfgrib/xarray
warnings.filterwarnings('ignore', category=UserWarning)

# ============================================================
# DIRECTORIOS DEL PROYECTO
# ============================================================

from pathlib import Path

PROJECT_ROOT = Path.cwd()

output_dir = PROJECT_ROOT / "data" / "processed" / "gfs"

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

print(output_dir)

# 1. Configuración Geográfica OAQ
LAT_OAQ = -0.2150
LON_OAQ = -78.5025

# 2. Parámetros de la simulación piloto
fecha_run = pd.Timestamp("2026-05-10 00:00")
lead_times = list(range(0, 25, 3)) # [0, 3, 6, 9, 12, 15, 18, 21, 24]

# 3. Diccionario de variables seleccionadas de la Subfase 3B
variables_gfs = {
    "RH700": ("RH", "700 mb"),
    "RH500": ("RH", "500 mb"),
    "T700": ("TMP", "700 mb"),
    "T500": ("TMP", "500 mb"),
    "U700": ("UGRD", "700 mb"),
    "V700": ("VGRD", "700 mb"),
    "Omega700": ("VVEL", "700 mb"),
    "PWAT": ("PWAT", None), # Corrección: es mejor definir explícitamente el nivel
    "TCC": ("TCDC", "entire atmosphere"),
    "HCC": ("HCDC", None)
}

c:\Users\Brian OAQ\Desktop\OAQ\2026\12. Manuscrito de un artículo Modelo predictivo de nubosidad para observaciones astronómicas\OAQ-AstroForecast\notebooks\03_gfs\data\processed\gfs


In [3]:
def extraer_variable_oaq(herbie_obj, variable, nivel=None):
    """
    Extrae e interpola linealmente una variable de Herbie sobre el OAQ
    manejando las excepciones de hypercubes y metadatos de proyección.
    """
    lon_oaq_gfs = LON_OAQ % 360
    patron = f":{variable}:" if nivel is None else f":{variable}:{nivel}:"

    try:
        ds = herbie_obj.xarray(patron)

        # Manejo de múltiples hypercubes devueltos como lista (Ej: TCC)
        if isinstance(ds, list):
            # Usar el primero o último que contenga la dimensión meteorológica real
            ds = ds[-1]

        # FILTRO CRÍTICO: Evitar seleccionar 'gribfile_projection' como variable
        vars_validas = [v for v in ds.data_vars if v != 'gribfile_projection']
        if not vars_validas:
            return np.nan
        nombre_var = vars_validas[0]
        
        da = ds[nombre_var]

        # Si persisten dimensiones duplicadas/huérfanas por step, tomamos el primer índice
        if 'step' in da.coords and da.ndim > 2:
            da = da.isel(step=0)

        # Interpolación bilineal
        valor_interpolado = da.interp(latitude=LAT_OAQ, longitude=lon_oaq_gfs, method='linear')
        return float(valor_interpolado.values)

    except Exception as e:
        print(f" Error en {variable} ({nivel}): {e}")
        return np.nan

In [4]:
# ============================================================
# PERÍODO DE DESCARGA
# ============================================================

fecha_inicio = pd.Timestamp("2026-01-01")
fecha_fin    = pd.Timestamp("2026-01-03")

horas_run = [0, 6, 12, 18]

runs = []

for fecha in pd.date_range(fecha_inicio, fecha_fin, freq="D"):

    for hora in horas_run:

        runs.append(
            fecha + pd.Timedelta(hours=hora)
        )

lead_times = list(range(0, 25, 3))

print(f"Runs generados: {len(runs)}")
print(f"Lead times    : {len(lead_times)}")
print(f"Filas esperadas: {len(runs) * len(lead_times)}")

Runs generados: 12
Lead times    : 9
Filas esperadas: 108


In [5]:
# ============================================================
# CONSTRUCCIÓN DEL DATASET GFS-OAQ
# ============================================================

gfs_dataset = []

for run_time in runs:

    run_time = pd.Timestamp(run_time)

    print(f"\nProcesando Run: {run_time}")

    for fxx in lead_times:

        print(f"   F{fxx:03d}")

        # Crear objeto Herbie para esta corrida y lead time
        H = Herbie(
            run_time,
            model="gfs",
            product="pgrb2.0p25",
            fxx=fxx
        )

        # Información temporal
        fila = {
            "run_time": run_time,
            "valid_time": run_time + pd.Timedelta(hours=fxx),
            "fxx": fxx
        }

        # Extraer variables atmosféricas
        for nombre, (variable, nivel) in variables_gfs.items():

            fila[nombre] = extraer_variable_oaq(
                H,
                variable=variable,
                nivel=nivel
            )

        gfs_dataset.append(fila)

# Construcción final del DataFrame
df_gfs_dataset = pd.DataFrame(gfs_dataset)

print("\nDataset construido correctamente.")
print(f"Filas: {len(df_gfs_dataset)}")



Procesando Run: 2026-01-01 00:00:00
   F000
✅ Found ┊ model=gfs ┊ product=pgrb2.0p25 ┊ 2026-Jan-01 00:00 UTC F00 ┊ GRIB2 @ aws ┊ IDX @ local
   F003
✅ Found ┊ model=gfs ┊ product=pgrb2.0p25 ┊ 2026-Jan-01 00:00 UTC F03 ┊ GRIB2 @ aws ┊ IDX @ local
Note: Returning a list of [2] xarray.Datasets because cfgrib opened with multiple hypercubes.
   F006
✅ Found ┊ model=gfs ┊ product=pgrb2.0p25 ┊ 2026-Jan-01 00:00 UTC F06 ┊ GRIB2 @ aws ┊ IDX @ local
Note: Returning a list of [2] xarray.Datasets because cfgrib opened with multiple hypercubes.
   F009
✅ Found ┊ model=gfs ┊ product=pgrb2.0p25 ┊ 2026-Jan-01 00:00 UTC F09 ┊ GRIB2 @ aws ┊ IDX @ local
Note: Returning a list of [2] xarray.Datasets because cfgrib opened with multiple hypercubes.
   F012
✅ Found ┊ model=gfs ┊ product=pgrb2.0p25 ┊ 2026-Jan-01 00:00 UTC F12 ┊ GRIB2 @ aws ┊ IDX @ local
Note: Returning a list of [2] xarray.Datasets because cfgrib opened with multiple hypercubes.
   F015
✅ Found ┊ model=gfs ┊ product=pgrb2.0p25 ┊ 2026-Jan-01

In [9]:
# ============================================================
# GUARDAR DATASET
# ============================================================

output_file = output_dir / "gfs_oaq_dataset.parquet"

df_gfs_dataset.to_parquet(
    output_file,
    index=False
)

print(output_file)
print(df_gfs_dataset.shape)

c:\Users\Brian OAQ\Desktop\OAQ\2026\12. Manuscrito de un artículo Modelo predictivo de nubosidad para observaciones astronómicas\OAQ-AstroForecast\notebooks\03_gfs\data\processed\gfs\gfs_oaq_dataset.parquet
(108, 13)
